<p><font size="6" color="grey"><b>
Generative KI. Verstehen. Anwenden. Gestalten.
</b></font></p>

<p><font size="5" color="grey"><b>
A00 | Hands-on Kopiervorlagen
</b></font></p>

---

# Hands-on statt Theorie

Dieses Notebook ist eine kompakte Copy/Paste-Sammlung für Google Colab. Es enthält nur lauffähige Grundmuster, die im Kurs schnell wiederverwendet werden können.

Ausführliche Erklärungen, Hintergründe und Best Practices stehen in `GenAI/docs/06-frameworks`. Diese Datei bleibt bewusst knapp, weil sie direkt in der Colab-Oberfläche verfügbar ist.

# 0 | Colab-Setup

In [ ]:
#@title Umgebung einrichten { display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/GenAI.git#subdirectory=04_modul

from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt,
)
from genai_lib.model_config import (
    BASELINE,
    ROUTER,
    JUDGE,
    PLANNER,
    WORKER,
    WORKER_PREMIUM,
    CODING,
    EMBEDDINGS,
)

setup_api_keys(["OPENAI_API_KEY"], create_globals=False)
check_environment()
print()
get_ipinfo()

## Optional | LangSmith-Tracing

In [ ]:
# Optional: Nur aktivieren, wenn LANGSMITH_API_KEY als Colab-Secret vorhanden ist.
import os

# setup_api_keys(["LANGSMITH_API_KEY"], create_globals=False)
# os.environ["LANGSMITH_TRACING"] = "true"
# os.environ["LANGSMITH_PROJECT"] = "A00-Snippets-GenAI"
# os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

## Standard-Imports

In [ ]:
from IPython.display import Markdown, display
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough

llm = init_chat_model(BASELINE)
parser = StrOutputParser()

# 1 | Minimaler LLM-Aufruf

In [ ]:
user_input = "Erkläre generative KI in einem Satz."
response = llm.invoke([HumanMessage(content=user_input)])
print(response.content)

# 2 | Prompt + Chain

In [ ]:
prompt = ChatPromptTemplate([
    ("system", "Du erklärst technische Themen knapp, korrekt und mit einem Beispiel."),
    ("human", "{user_input}"),
])

chain = prompt | llm | StrOutputParser()
result = chain.invoke({"user_input": "Was ist Retrieval-Augmented Generation?"})
print(result)

# 3 | Kurs-Prompt laden

In [ ]:
url = "https://github.com/ralf-42/GenAI/blob/main/05_prompt/create_prompt.md"
chat_template = load_prompt(url)
chain = chat_template | llm | StrOutputParser()

response = chain.invoke({
    "context": (
        "Wir entwickeln eine API für Zahlungsabwicklung. "
        "Die Dokumentation ist veraltet. "
        "Zielgruppe: Full-Stack Developer ohne Payment-Erfahrung."
    ),
    "role": "Du bist ein erfahrener Technical Writer für Developer-Dokumentation.",
    "expectation": (
        "Erstelle eine Übersichtsseite mit maximal 500 Wörtern. "
        "Erkläre die Endpunkte /create-payment, /verify-payment und /refund."
    ),
    "action_plan": (
        "1. Zielgruppe analysieren\n"
        "2. Struktur festlegen\n"
        "3. Fachbegriffe erklären\n"
        "4. Praktische Beispiele hinzufügen"
    ),
    "tone": "Professionell, zugänglich, ohne Marketing-Sprache.",
    "evaluation": "Die Antwort ist klar strukturiert und für Developer direkt nutzbar.",
})

print(response)

## Mehrere Aufgaben mit demselben Kurs-Prompt

In [ ]:
base_params = {
    "context": "Eine bestehende Payment-API braucht bessere Dokumentation.",
    "role": "Du bist ein erfahrener Technical Writer für Developer-Dokumentation.",
    "action_plan": "Analysieren, strukturieren, erklären, Beispiel ergänzen.",
    "tone": "Klar, technisch, pragmatisch.",
    "evaluation": "Die Ausgabe ist konkret und wiederverwendbar.",
}

expectations = [
    "Erstelle eine kurze API-Übersicht.",
    "Erstelle ein Beispiel für einen erfolgreichen Payment-Flow.",
    "Erstelle eine Troubleshooting-Liste für typische Fehler.",
]

for expectation in expectations:
    result = chain.invoke({**base_params, "expectation": expectation})
    display(Markdown(f"## {expectation}\n\n{result}"))

# 4 | Strukturierte Ausgabe

In [ ]:
from pydantic import BaseModel, Field

class QuestionAnswer(BaseModel):
    question: str = Field(description="Die gestellte Frage")
    answer: str = Field(description="Die Antwort auf die Frage")
    confidence: int = Field(description="Vertrauen in die Antwort von 0 bis 100")
    category: str = Field(description="Kategorie der Frage")

structured_llm = llm.with_structured_output(QuestionAnswer)
structured_chain = ChatPromptTemplate([
    ("system", "Beantworte die Frage strukturiert."),
    ("human", "{question}"),
]) | structured_llm

result = structured_chain.invoke({"question": "Was ist Photosynthese?"})
print(result)
print(result.answer)

# 5 | Chat-History

In [ ]:
chat_history = []

history_chain = (
    ChatPromptTemplate([
        ("system", "Du bist ein hilfreicher Assistent."),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{user_input}"),
    ])
    | llm
    | StrOutputParser()
)

user_input = "Mein Name ist Max."
response = history_chain.invoke({"chat_history": chat_history, "user_input": user_input})
chat_history.extend([HumanMessage(content=user_input), AIMessage(content=response)])
print(response)

user_input = "Wie heiße ich?"
response = history_chain.invoke({"chat_history": chat_history, "user_input": user_input})
print(response)

## Automatisches Memory-Management

In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

session_store = {}

def get_session_history(session_id):
    if session_id not in session_store:
        session_store[session_id] = InMemoryChatMessageHistory()
    return session_store[session_id]

prompt_with_history = ChatPromptTemplate([
    ("system", "Du bist ein hilfreicher Assistent."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{user_input}"),
])

chain_with_history = RunnableWithMessageHistory(
    prompt_with_history | llm | StrOutputParser(),
    get_session_history=get_session_history,
    input_messages_key="user_input",
    history_messages_key="history",
)

config = {"configurable": {"session_id": "kurs-demo"}}
print(chain_with_history.invoke({"user_input": "Ich lerne gerade LangChain."}, config=config))
print(chain_with_history.invoke({"user_input": "Was lerne ich gerade?"}, config=config))

# 6 | RAG-Kopiervorlage

In [ ]:
# Falls noch nicht vorhanden:
# !uv pip install --system -q langchain-community langchain-openai chromadb pypdf unstructured

from langchain import hub
from langchain_chroma import Chroma
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader, TextLoader, UnstructuredMarkdownLoader
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

source_directory = "/content/files"
persistent_directory = "/content/chroma_db"

loaders = [
    DirectoryLoader(source_directory, glob="**/*.md", loader_cls=UnstructuredMarkdownLoader),
    DirectoryLoader(source_directory, glob="**/*.txt", loader_cls=TextLoader),
    DirectoryLoader(source_directory, glob="**/*.pdf", loader_cls=PyPDFLoader),
]

documents = []
for loader in loaders:
    try:
        documents.extend(loader.load())
    except Exception as exc:
        print(f"Loader übersprungen: {exc}")

text_splitter = RecursiveCharacterTextSplitter(chunk_size=900, chunk_overlap=150)
docs = text_splitter.split_documents(documents)

embeddings = OpenAIEmbeddings(model=EMBEDDINGS)
vectorstore = Chroma.from_documents(docs, embeddings, persist_directory=persistent_directory)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

## RAG-Aufruf

In [ ]:
rag_prompt = hub.pull("rlm/rag-prompt")

def format_documents(documents):
    return "\n\n".join(doc.page_content for doc in documents)

rag_chain = (
    {"context": retriever | format_documents, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

question = "Welche Kernaussagen stehen in den Dokumenten?"
answer = rag_chain.invoke(question)
print(answer)

# 7 | SQL-RAG-Kopiervorlage

In [ ]:
# Beispiel erwartet eine SQLite-Datei unter /content/northwind.db
import re
import sqlite3
from langchain_community.utilities import SQLDatabase

DB_PATH = "/content/northwind.db"
DB_URI = f"sqlite:///{DB_PATH}"
db = SQLDatabase.from_uri(DB_URI)

sql_prompt = PromptTemplate.from_template("""
Du bist ein SQL-Experte. Erzeuge eine SQLite-Abfrage für die Benutzerfrage.
Nutze ausschließlich Tabellen und Spalten aus dem Schema.
Gib nur SQL aus, ohne Markdown und ohne Erklärung.
Begrenze Listen auf maximal 10 Zeilen.

Schema:
{schema}

Frage: {query}
SQL:
""")

analysis_prompt = PromptTemplate.from_template("""
Beantworte die Benutzerfrage anhand der SQL-Ergebnisse kurz und verständlich.

Frage: {query}
SQL: {sql_query}
Ergebnisse:
{results}

Antwort:
""")

def get_schema(_):
    return db.get_table_info()

def clean_sql(sql_query):
    sql_query = re.sub(r"```sql\s*(.*?)\s*```", r"\1", sql_query, flags=re.DOTALL)
    return sql_query.replace("```", "").strip()

def execute_query(sql_query):
    sql_query = clean_sql(sql_query)
    with sqlite3.connect(DB_PATH) as conn:
        cursor = conn.cursor()
        cursor.execute(sql_query)
        rows = cursor.fetchall()
        columns = [description[0] for description in cursor.description]
    table = "| " + " | ".join(columns) + " |\n"
    table += "| " + " | ".join(["---"] * len(columns)) + " |\n"
    table += "".join("| " + " | ".join(map(str, row)) + " |\n" for row in rows)
    return table or "Keine Ergebnisse gefunden."

sql_generator = (
    RunnablePassthrough.assign(schema=get_schema)
    | sql_prompt
    | llm
    | StrOutputParser()
)

analysis_chain = analysis_prompt | llm | StrOutputParser()

def ask_database(query):
    sql_query = clean_sql(sql_generator.invoke({"query": query}))
    results = execute_query(sql_query)
    answer = analysis_chain.invoke({"query": query, "sql_query": sql_query, "results": results})
    return f"### SQL\n```sql\n{sql_query}\n```\n\n### Ergebnisse\n{results}\n\n### Antwort\n{answer}"

# display(Markdown(ask_database("Welche Produkte haben den höchsten Umsatz?")))

# 8 | Agent mit Tools

In [ ]:
from langchain.agents import create_agent
from langchain_core.tools import tool

@tool
def multiply(a: int, b: int) -> int:
    """Multipliziert zwei Zahlen."""
    return a * b

@tool
def add(a: int, b: int) -> int:
    """Addiert zwei Zahlen."""
    return a + b

@tool
def divide(a: float, b: float) -> float:
    """Dividiert zwei Zahlen. Gibt einen Fehler zurück, wenn b gleich 0 ist."""
    if b == 0:
        return "Fehler: Division durch Null"
    return a / b

tools = [multiply, add, divide]
agent = create_agent(
    model=init_chat_model(WORKER),
    tools=tools,
    system_prompt="Du bist ein Mathe-Assistent. Nutze Tools für Berechnungen.",
)

response = agent.invoke({
    "messages": [{"role": "user", "content": "Was ist 25 mal 4 plus 7?"}]
})

print(response["messages"][-1].content)

## Agent-Middleware und Cache-Control

In [ ]:
from langchain.agents.middleware import HumanInTheLoopMiddleware, SummarizationMiddleware, PIIMiddleware

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Sendet eine E-Mail. Demo-Tool: verschickt nichts wirklich."""
    return f"E-Mail an {to} würde gesendet: {subject}"

middleware = [
    HumanInTheLoopMiddleware(tool_names=["send_email"]),
    SummarizationMiddleware(max_tokens=1000),
    PIIMiddleware(
        patterns={"email": r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}"},
        redact=True,
    ),
]

system_message = SystemMessage(
    content="Du bist ein hilfreicher Kurs-Assistent.",
    cache_control={"type": "ephemeral"},
)

# Je nach Runtime/Provider kann Middleware zusätzliche Konfiguration brauchen.
# middleware_agent = create_agent(
#     model=init_chat_model(WORKER),
#     tools=[send_email],
#     middleware=middleware,
#     system_prompt=system_message,
# )

# 9 | Kleine Helfer

In [ ]:
profile = get_model_profile(BASELINE)
print(profile)

In [ ]:
diagram = """
graph TD
    A[Frage] --> B[Retriever]
    B --> C[Kontext]
    C --> D[Prompt]
    D --> E[LLM]
    E --> F[Antwort]
"""
mermaid(diagram, width=700, height=420)